# Spatial Index Notebook (general) with utilities module

### Goals: (1) Generate and save colonies dataset (2) Create function that creates service index. (3) Apply service index function to all available services (4) Add overall PSI column. (5) Save/ship to colleagues

In this notebook, I will complete the following tasks:
* Load in `colonies_with_neighbors` and process **[DONE]**
    * Merge population data with colonies data
    * Create nbr_dist column
    * Add ndmc_dist column
    * Remove extraneous columns and save as pickle file
* Calculate and add services indices to GeoDataFrame
    * Where needed, convert CSV lat/long to GeoDataFrame with Shapely Points
    * Convert shapefiles to the same coordinate reference system
    * Check that all point coordinates are in Delhi
    * Make sure that no geometry fields are missing
    * Calculate service index and add to GeoDataFrame
* Descriptive statistics for service indices
    * mean, min, max
    * Grouped by settlement type: take average PSI for all colonies with a specific settlement type
    * Grouped by MCD category: take average PSI for all colonies with a specific MCD category
    * Grouped by distance from NDMC? (optional)
* Save service indices and descriptive statistics to .csv file

In [ ]:
# Import necessary modules
from itertools import islice
import pickle
from importlib import reload
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from shapely.geometry import box, Polygon, Point
from shapely.ops import cascaded_union
from pyproj import CRS
import spatial_index_utils

# Reload spatial_index_utils
reload(spatial_index_utils)

# Constants
# Pseudo Mercator
epsg_code = 3857 

## Process and save colonies shapefile for PSI

In [ ]:
# open colonies_with_neighbors file
# This has NDMC+JJC colonies with neighbors of each polygon
# based on other polygons that intersect, cross, touch
# or overlap.

with open('colonies_with_neighbors.data', 'rb') as f:
    colonies = pickle.load(f)

In [ ]:
# Reproject to EPSG 3857
colonies = spatial_index_utils.reproject_gdf(colonies, epsg_code)

In [ ]:
colonies.head()

In [ ]:
## Calculate distances from polygon centroid to centroids of its neighbors

# Distance should be in meters
# https://en.wikipedia.org/wiki/Easting_and_northing

colonies = spatial_index_utils.calc_nbr_dist(colonies)

In [ ]:
# Import 2020 population data
worldpop2020 = pd.read_csv("population_data/pop_colony_wp_2020.csv")

# Restrict dataframe to only two columns:
# layer: population data
# uso_area_u: unique id for colonies
worldpop2020 = worldpop2020[['layer', 'uso_area_u']]

# Merge population data with colonies data
colonies = colonies.merge(worldpop2020, how='inner', 
                          left_on="USO_AREA_U", right_on='uso_area_u')

# Rename 'layer' column as 'population'
colonies = colonies.rename(columns={'layer': 'population'})

# Remove additional column 'uso_area_u'
colonies = colonies.drop(columns=['uso_area_u'])

In [ ]:
# Code to generate ndmc_distances (in kilometers)

## 28.632846 77.219639 Rajiv Chowk is considered the center of the city. 
# Cities of Delhi used used this same notional center.
# EPSG3857, WGS84

colonies['ndmc_dist'] = 0
ndmc = Point(28.632846, 77.219639)
for idx, row in colonies.iterrows():
    colonies.loc[idx, 'ndmc_dist'] = ndmc.distance(row['centroid'])/1000.0

In [ ]:
# Remove centroid and polygon_neighbors columns, which
# are no longer needed now that we have nbr_dist
colonies = colonies.drop(columns=['centroid', 'polygon_neighbors'])

In [ ]:
colonies.head(2)

In [ ]:
## Save `colonies` as pickle file
with open('colonies_for_psi.data', 'wb') as f:
    pickle.dump(colonies, f)

## START HERE: Load colonies GeoDataFrame for PSI

In [ ]:
with open('colonies_for_psi.data', 'rb') as f:
    colonies = pickle.load(f)

In [ ]:
colonies.head(2)

## Load Ration Shop and check for missing geometries or duplicate rows

In [ ]:
# Read in ration shop data
ration_shops = gpd.read_file('RationShops.shp')

# Make sure that all ration shops have a geometry
ration_shops[ration_shops['geometry'] == None]

In [ ]:
# Check for duplicate rows
ration_shops[ration_shops.duplicated()]

## TODO: 
* Check for geometry duplicates
* Check that all geometries are in Delhi
* Create function `check_shapefile`
* Duplicates
    * be more selected in checking for duplicates? Check for most minimal things
    * be restricted in what I am checking out - map no, reg no, and geometry.
    * print this out...
* Unauthorized colonies

## Convert ration shop index creation into function 

In [ ]:
colonies = spatial_index_utils.create_service_index(polygon_gdf=colonies, 
                                                    point_gdf=ration_shops, 
                                                    service_name="ration", 
                                                    epsg_code=epsg_code)

In [ ]:
colonies.head()

## Load in Schools Dataset and Generate Index

In [ ]:
# Read in schools data
schools = gpd.read_file('DelhiSchoolsMerged.shp')

# Make sure that all ration shops have a geometry
schools[schools['geometry'] == None]

In [ ]:
# Check for duplicate rows
schools[schools.duplicated()]

In [ ]:
colonies = spatial_index_utils.create_service_index(polygon_gdf=colonies, 
                                                    point_gdf=schools, 
                                                    service_name="school", 
                                                    epsg_code=epsg_code)

In [ ]:
colonies.head()

## Convert CSV data into shapefiles

In [ ]:
# Delhi constants
lat_min = 28.412593
lat_max = 28.881338
lon_min = 76.83806899999999
lon_max = 77.3484578

In [ ]:
delhi_bbox = box(lon_min, lat_min, lon_max, lat_max)

### ATM dataset

In [ ]:
atm = gpd.read_file('atm_wgs84.shp')

In [ ]:
atm.crs

In [ ]:
len(atm)

In [ ]:
atm = atm[atm.intersects(delhi_bbox)]

In [ ]:
len(atm)

In [ ]:
# Make sure that all have a geometry
atm[atm['geometry'] == None]

In [ ]:
# Check for duplicate rows
atm[atm.duplicated()]

In [ ]:
colonies = spatial_index_utils.create_service_index(polygon_gdf=colonies, 
                                                    point_gdf=atm, 
                                                    service_name="atm", 
                                                    epsg_code=epsg_code)

In [ ]:
colonies.head(2)

In [ ]:
colonies['atm_idx'].max()

### Bank Index

In [ ]:
bank = gpd.read_file('bank_wgs84.shp')

In [ ]:
bank.head()

In [ ]:
bank.crs

In [ ]:
#bank = bank[bank.intersects(delhi_bbox)]

In [ ]:
# Make sure that all have a geometry
bank[bank['geometry'] == None]

In [ ]:
bank[bank.duplicated()]

In [ ]:
colonies = spatial_index_utils.create_service_index(polygon_gdf=colonies, 
                                                    point_gdf=bank, 
                                                    service_name="bank", 
                                                    epsg_code=epsg_code)

In [ ]:
colonies.head()

### Metro

In [ ]:
metro = gpd.read_file('metro_wgs84.shp')

In [ ]:
metro.head()

In [ ]:
# Make sure that all have a geometry
metro[metro['geometry'] == None]

In [ ]:
# Make sure none are duplicated
metro[metro.duplicated()]

In [ ]:
colonies = spatial_index_utils.create_service_index(polygon_gdf=colonies, 
                                                    point_gdf=metro, 
                                                    service_name="metro", 
                                                    epsg_code=epsg_code)

In [ ]:
colonies.head()

### Police

In [ ]:
police = gpd.read_file('police_wgs84.shp')

In [ ]:
police.head()

In [ ]:
# Make sure that all have a geometry
police[police['geometry'] == None]

In [ ]:
# Make sure none are duplicated
police[police.duplicated()]

In [ ]:
colonies = spatial_index_utils.create_service_index(polygon_gdf=colonies, 
                                                    point_gdf=police, 
                                                    service_name="police", 
                                                    epsg_code=epsg_code)

In [ ]:
colonies.head()

### Bus

In [ ]:
bus = gpd.read_file('bus.shp')

In [ ]:
bus.head()

In [ ]:
len(bus)

In [ ]:
# Make sure that all have a geometry
bus[bus['geometry'] == None]

In [ ]:
# Make sure none are duplicated
bus[bus.duplicated()]

In [ ]:
colonies = spatial_index_utils.create_service_index(polygon_gdf=colonies, 
                                                    point_gdf=bus, 
                                                    service_name="bus", 
                                                    epsg_code=epsg_code)

In [ ]:
colonies.head()

## Add area column

In [ ]:
colonies['area_m2'] = colonies['geometry'].area

In [ ]:
colonies.head()

## Rename `ndmc_dist` to `ndmc_dist_km` to reflect unit

In [ ]:
colonies = colonies.rename(columns={'ndmc_dist':'ndmc_dist_km'})

In [ ]:
colonies.head()

## Save ration shop index (and descriptive data) to CSV and pickle files

In [ ]:
colonies.to_csv('colonies_psi_25july.csv')

In [ ]:
with open('colonies_psi_25july.data', 'wb') as f:
    pickle.dump(colonies, f)